In [ ]:
import pandas as pd
import os

def load_parquet_dataset(dataset_name, data_dir="./data"):
    """
    Charge un dataset au format Parquet depuis le dossier de données.

    Paramètres:
    - dataset_name (str): Nom du dataset (ex: 'ragbench', 'hotpotqa').
                          Peut inclure ou non l'extension '.parquet'.
    - data_dir (str): Chemin vers le dossier contenant les fichiers (défaut: './data').

    Retourne:
    - pd.DataFrame: Le dataframe prêt à l'emploi.
    """
    # 1. Nettoyage du nom (ajoute .parquet si oublié)
    if not dataset_name.endswith('.parquet'):
        file_name = f"{dataset_name}.parquet"
    else:
        file_name = dataset_name

    # 2. Construction du chemin sécurisé (compatible Windows/Linux/Mac)
    file_path = os.path.join(data_dir, file_name)

    # 3. Sécurité : Vérification de l'existence du fichier
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Erreur : Le fichier est introuvable au chemin -> {file_path}")

    # 4. Chargement via Pandas
    print(f"Chargement de {file_name} en cours...")
    df = pd.read_parquet(file_path)
    print(f"-> Succès : {len(df)} lignes chargées.")

    return df

# Approches 3 et 4

In [ ]:
import re
import gc
import torch
import pandas as pd

# ==========================================
# 1. Utilitaires techniques
# ==========================================
def clean_memory():
    """Vide le cache mémoire GPU et force le Garbage Collector."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def generate_llm_response(model, tokenizer, prompt, max_tokens=350):
    """Gère l'inférence du LLM de manière isolée."""
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_tokens,
            do_sample=False
        )

    raw_output = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    del inputs, outputs
    clean_memory()

    return raw_output

# ==========================================
# 2. Parsing et Métriques
# ==========================================
def process_output_with_tags(raw_output):
    """Extrait la réponse textuelle et la liste d'IDs depuis les balises XML."""
    answer_match = re.search(r"<answer>(.*?)</answer>", raw_output, re.DOTALL)
    answer = answer_match.group(1).strip() if answer_match else "Format Error (Answer)"

    chunks_match = re.search(r"<chunks_id>(.*?)</chunks_id>", raw_output, re.DOTALL)
    predicted_ids = []
    if chunks_match:
        predicted_ids = [int(n) for n in re.findall(r'\d+', chunks_match.group(1))]

    return answer, predicted_ids

def calculate_metrics(gold_ids, predicted_ids):
    """Calcule P, R, F1 et Accuracy (Exact Match)."""
    gold_set = set(gold_ids)
    pred_set = set(predicted_ids)

    correct_ids = pred_set.intersection(gold_set)

    # Précision & Rappel
    p = len(correct_ids) / len(pred_set) if len(pred_set) > 0 else 0.0
    r = len(correct_ids) / len(gold_set) if len(gold_set) > 0 else 0.0

    # F1-Score
    f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0.0

    # Accuracy (définie ici comme Exact Match: la liste prédite doit être exactement la liste gold)
    accuracy = 1.0 if pred_set == gold_set else 0.0

    return {"precision": p, "recall": r, "f1": f1, "accuracy": accuracy}

# ==========================================
# 3. Le Contrôleur Principal
# ==========================================
def evaluate_single_prompt_approach(dataset, prompt_template, model, tokenizer, sample_size=None, output_filename=None):
    """
    Évalue un dataset avec une approche à prompt unique (ex: Approches 3 et 4).

    :param dataset: Le DataFrame (pandas) contenant les données
    :param prompt_template: Le prompt formaté (ex: cite_then_answer ou answer_then_cite)
    :param model: Le modèle LLM chargé
    :param tokenizer: Le tokenizer associé
    :param sample_size: Nombre de lignes à tester (équivalent de head()). Si None, lance tout.
    :param output_filename: Nom du fichier CSV pour sauvegarder les résultats (ex: "results_app3.csv"). Si None, pas de sauvegarde.
    """
    results = []

    # Sélection de l'échantillon dynamique
    df_sample = dataset.head(sample_size) if sample_size is not None else dataset

    for index, row in df_sample.iterrows():
        # 1. Préparation de l'input
        context_text = "\n".join([f"ID: {p['id']} | Paragraph: {p['text']}" for p in row['contexts']])
        full_prompt = prompt_template.format(
            context_text=context_text,
            question=row['question']
        )

        # 2. Inférence
        raw_output = generate_llm_response(model, tokenizer, full_prompt)

        # 3. Parsing
        predicted_answer, predicted_ids = process_output_with_tags(raw_output)

        # 4. Évaluation
        metrics = calculate_metrics(row['gold_ids'], predicted_ids)

        # 5. Sauvegarde des données
        results.append({
            "sample_id": row['id'],
            "question": row['question'],
            "gold_ids": row['gold_ids'],
            "predicted_ids": predicted_ids,
            "predicted_answer": predicted_answer,
            **metrics
        })

        # 6. Affichage visuel par itération
        print(f"--- SAMPLE {index+1} (ID: {row['id']}) ---")
        print(f"Question : {row['question']}")
        print(f"Réponse du LLM : {predicted_answer}")
        print(f"Gold IDs : {row['gold_ids']}")
        print(f"Pred IDs : {predicted_ids}")
        print(f"Verdict : {'✅ Exact' if metrics['accuracy'] else '❌ Différent'} (P: {metrics['precision']:.2f}, R: {metrics['recall']:.2f}, F1: {metrics['f1']:.2f})")
        print("-" * 50)

    # Agrégation des résultats finaux
    df_results = pd.DataFrame(results)
    avg_metrics = df_results[["precision", "recall", "f1", "accuracy"]].mean().to_dict()

    print("\n" + "="*20 + " RÉSULTATS GLOBAUX " + "="*20)
    for k, v in avg_metrics.items():
        print(f"Moyenne {k.capitalize()} : {v:.4f}")
    print("="*59 + "\n")

    # Sauvegarde conditionnelle
    if output_filename:
        df_results.to_csv(output_filename, index=False, encoding='utf-8')
        print(f" Résultats sauvegardés avec succès dans : {output_filename}")

    return df_results, avg_metrics

In [ ]:
import re
import pandas as pd

# ==========================================
# 1. Nouveaux Parseurs Spécifiques (NLI)
# ==========================================
def extract_atomic_claims(raw_output):
    """Extrait et nettoie la liste des claims depuis la balise <atomic_claims>."""
    claims_match = re.search(r"<atomic_claims>(.*?)</atomic_claims>", raw_output, re.DOTALL)
    claims_list = []

    if claims_match:
        raw_claims_text = claims_match.group(1).strip()
        # On sépare par les tirets en début de ligne ou simplement les tirets
        raw_claims = raw_claims_text.split('\n-')
        for claim in raw_claims:
            # On nettoie les tirets restants et les espaces
            cleaned = claim.strip().lstrip('-').strip()
            if cleaned:
                claims_list.append(cleaned)

    return claims_list

def extract_nli_verdict(raw_output):
    """Extrait l'ID et le LABEL depuis la balise <nli_verdict>."""
    verdict_match = re.search(r"<nli_verdict>(.*?)</nli_verdict>", raw_output, re.DOTALL)

    if verdict_match:
        content = verdict_match.group(1)
        # Extraction de l'ID
        id_match = re.search(r"ID:\s*(\d+)", content)
        chunk_id = int(id_match.group(1)) if id_match else None

        # Extraction du Label
        label_match = re.search(r"LABEL:\s*(ENTAILMENT|NEUTRAL|CONTRADICTION)", content, re.IGNORECASE)
        label = label_match.group(1).upper() if label_match else "UNKNOWN"

        return chunk_id, label

    return None, "FORMAT_ERROR"

# ==========================================
# 2. Contrôleur Approche 5 (NLI)
# ==========================================
def evaluate_approach_5(dataset, prompt_answer, prompt_claims, prompt_nli, model, tokenizer, sample_size=None, output_filename=None):
    """
    Évalue l'Approche 5 : LLM génère la réponse -> LLM extrait les claims -> LLM vérifie Claim vs Chunk (N x M).
    """
    results = []
    df_sample = dataset.head(sample_size) if sample_size is not None else dataset

    for index, row in df_sample.iterrows():
        # --- ÉTAPE A : Génération de la Réponse ---
        context_text = "\n".join([f"ID: {p['id']} | Paragraph: {p['text']}" for p in row['contexts']])

        prompt_1 = prompt_answer.format(
            context_text=context_text,
            question=row['question']
        )
        raw_output_1 = generate_llm_response(model, tokenizer, prompt_1)
        predicted_answer = extract_answer(raw_output_1)

        # --- ÉTAPE B : Décomposition en Atomic Claims ---
        prompt_2 = prompt_claims.format(
            predicted_answer=predicted_answer
        )
        raw_output_2 = generate_llm_response(model, tokenizer, prompt_2)
        atomic_claims = extract_atomic_claims(raw_output_2)

        # --- ÉTAPE C : Vérification NLI (Boucle N x M) ---
        predicted_ids_set = set() # Utilisation d'un set pour éviter les doublons d'IDs
        detailed_verdicts = []    # Optionnel: pour garder une trace des logs complets

        for chunk in row['contexts']:
            for claim in atomic_claims:
                prompt_3 = prompt_nli.format(
                    chunk_id=chunk['id'],
                    source_chunk=chunk['text'],
                    atomic_claim=claim
                )

                raw_output_3 = generate_llm_response(model, tokenizer, prompt_3)
                v_id, v_label = extract_nli_verdict(raw_output_3)

                detailed_verdicts.append({"chunk_id": chunk['id'], "claim": claim, "label": v_label})

                # Règle d'agrégation : On garde l'ID si au moins UN claim est "ENTAILMENT"
                if v_label == "ENTAILMENT" and v_id is not None:
                    predicted_ids_set.add(v_id)

        # Conversion du set final en liste triée
        predicted_ids = sorted(list(predicted_ids_set))

        # --- ÉVALUATION ET SAUVEGARDE ---
        metrics = calculate_metrics(row['gold_ids'], predicted_ids)

        results.append({
            "sample_id": row['id'],
            "question": row['question'],
            "gold_ids": row['gold_ids'],
            "predicted_ids": predicted_ids,
            "predicted_answer": predicted_answer,
            "atomic_claims": atomic_claims,     # On sauvegarde les claims pour analyse
            "detailed_verdicts": detailed_verdicts, # Et la matrice des verdicts complets
            **metrics
        })

        # Affichage visuel
        print(f"--- SAMPLE {index+1} (ID: {row['id']}) ---")
        print(f"Question : {row['question']}")
        print(f"Réponse  : {predicted_answer}")
        print(f"Nb Claims: {len(atomic_claims)} extraits")
        print(f"Gold IDs : {row['gold_ids']}")
        print(f"Pred IDs : {predicted_ids}")
        print(f"Verdict  : {'✅ Exact' if metrics['accuracy'] else '❌ Différent'} (P: {metrics['precision']:.2f}, R: {metrics['recall']:.2f}, F1: {metrics['f1']:.2f})")
        print("-" * 50)

    # Agrégation des résultats finaux
    df_results = pd.DataFrame(results)
    avg_metrics = df_results[["precision", "recall", "f1", "accuracy"]].mean().to_dict()

    print("\n" + "="*20 + " RÉSULTATS APPROCHE 5 (NLI) " + "="*20)
    for k, v in avg_metrics.items():
        print(f"Moyenne {k.capitalize()} : {v:.4f}")
    print("="*68 + "\n")

    if output_filename:
        df_results.to_csv(output_filename, index=False, encoding='utf-8')
        print(f" Résultats sauvegardés dans : {output_filename}")

    return df_results, avg_metrics

In [ ]:
# Lancement de l'évaluation pour l'Approche 5 (NLI)
df_res_app5, metrics_app5 = evaluate_approach_5(
    dataset=ds_rag,
    prompt_answer=answer_for_nli,  # Prompt 1 : Génère la réponse
    prompt_claims=atomic_claims,   # Prompt 2 : Extrait les claims
    prompt_nli=nli_check,          # Prompt 3 : Juge Entailment
    model=model,
    tokenizer=tokenizer,
    sample_size=10,                # ⚠️ Attention, mets un petit chiffre au début (ex: 5 ou 10)
    output_filename="resultats_approche5_nli_llama3.csv"
)

# Optionnel : Afficher les résultats
display(df_res_app5.head())

# Appel API

In [ ]:
from openai import OpenAI

# 1. Configuration du client API
client = OpenAI(
    api_key="TA_CLE_API",          # Ta clé
    base_url="URL_DE_TON_SERVEUR"  # Ex: "https://serveur-llm.ton-entreprise.com/v1"
)
MODEL_NAME = "nom-du-modele"       # Ex: "Qwen3.6-35B-A3B-FP8"

# 2. La nouvelle fonction de génération
def generate_llm_response(model_client, tokenizer, prompt, max_tokens=350):
    """
    On ignore 'tokenizer' ici car l'API gère la tokenization en interne.
    On remplace 'model' par le 'model_client'.
    """
    response = model_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0.0 # Pour avoir des réponses déterministes
    )

    return response.choices[0].message.content

# 3. Plus besoin de clean_memory() !
def clean_memory():
    pass

#VLLM

In [ ]:
from vllm import LLM, SamplingParams

# 1. Chargement du modèle avec vLLM
def load_vllm_model(model_path="/chemin/vers/le/modele/cloné"):
    """
    Charge le modèle et son tokenizer via le moteur vLLM.
    """
    # tensor_parallel_size = 1 (utiliser 1 GPU).
    # Si ton modèle 35B nécessite 2 ou 4 GPUs pour tenir en mémoire, tu mets 2 ou 4.
    model = LLM(
        model=model_path,
        trust_remote_code=True,
        tensor_parallel_size=1
    )

    # vLLM charge automatiquement le tokenizer associé au modèle
    tokenizer = model.get_tokenizer()

    return model, tokenizer

# 2. Inférence avec vLLM
def generate_llm_response(model, tokenizer, prompt, max_tokens=350):
    """
    Génère du texte de manière ultra-rapide avec vLLM.
    """
    # A. Définition des paramètres (équivalent de do_sample=False)
    sampling_params = SamplingParams(
        temperature=0.0,
        max_tokens=max_tokens
    )

    # B. Formatage du prompt avec le template du modèle (comme on le faisait avec Transformers)
    messages = [{"role": "user", "content": prompt}]
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False, # Important : vLLM veut du texte ici, pas des tenseurs
        add_generation_prompt=True
    )

    # C. Génération (use_tqdm=False permet de ne pas polluer ta console avec des barres de chargement)
    outputs = model.generate([formatted_prompt], sampling_params, use_tqdm=False)

    # D. Extraction du texte généré
    raw_output = outputs[0].outputs[0].text

    return raw_output

# 3. Désactivation du nettoyeur de mémoire
def clean_memory():
    """
    vLLM gère la mémoire dynamiquement via PagedAttention.
    On laisse cette fonction vide (pass) pour ne pas casser les autres scripts
    qui essaieraient de l'appeler.
    """
    pass

In [ ]:
# exemple
# Test de l'approche 3 avec sauvegarde
df_res_app3, metrics_app3 = evaluate_single_prompt_approach(
    dataset=ds_rag,
    prompt_template=cite_then_answer,
    model=model,
    tokenizer=tokenizer,
    sample_size=20,
    output_filename="resultats_approche3_llama3.csv"
)

# Approche 1

In [ ]:
import re
import pandas as pd

# ==========================================
# 1. Nouveaux Parseurs Spécifiques
# ==========================================
def extract_ids(raw_output):
    """Extrait uniquement la liste des IDs depuis la balise <chunks_id>."""
    chunks_match = re.search(r"<chunks_id>(.*?)</chunks_id>", raw_output, re.DOTALL)
    if chunks_match:
        return [int(n) for n in re.findall(r'\d+', chunks_match.group(1))]
    return []

def extract_answer(raw_output):
    """Extrait uniquement le texte depuis la balise <answer>."""
    answer_match = re.search(r"<answer>(.*?)</answer>", raw_output, re.DOTALL)
    return answer_match.group(1).strip() if answer_match else "Format Error (Answer)"

# ==========================================
# 2. Logique de Filtrage (Spécifique Approche 1)
# ==========================================
def filter_context_by_ids(contexts, predicted_ids):
    """
    Filtre la liste de dictionnaires 'contexts' pour ne garder que ceux dont
    l'ID est dans 'predicted_ids'. Renvoie une string formatée.
    """
    if not predicted_ids:
        return "No evidence selected."

    filtered_texts = []
    for p in contexts:
        if p['id'] in predicted_ids:
            filtered_texts.append(f"ID: {p['id']} | Paragraph: {p['text']}")

    # Si le LLM a halluciné des IDs qui n'existent pas dans le contexte initial
    if not filtered_texts:
        return "Selected IDs do not match any available context."

    return "\n".join(filtered_texts)

# ==========================================
# 3. Contrôleur Approche 1
# ==========================================
def evaluate_approach_1(dataset, prompt_chunk_before, prompt_answer_before, model, tokenizer, sample_size=None, output_filename=None):
    """
    Évalue l'Approche 1 : LLM 1 extrait les sources -> Python filtre -> LLM 2 répond.
    """
    results = []
    df_sample = dataset.head(sample_size) if sample_size is not None else dataset

    for index, row in df_sample.iterrows():
        # --- ÉTAPE 1 : Identification des Chunks ---
        context_text = "\n".join([f"ID: {p['id']} | Paragraph: {p['text']}" for p in row['contexts']])
        prompt_1 = prompt_chunk_before.format(
            context_text=context_text,
            question=row['question']
        )

        raw_output_1 = generate_llm_response(model, tokenizer, prompt_1)
        predicted_ids = extract_ids(raw_output_1)

        # --- ÉTAPE INTERMÉDIAIRE : Filtrage ---
        filtered_context = filter_context_by_ids(row['contexts'], predicted_ids)

        # --- ÉTAPE 2 : Génération de la Réponse ---
        prompt_2 = prompt_answer_before.format(
            filtered_context=filtered_context,
            question=row['question']
        )

        raw_output_2 = generate_llm_response(model, tokenizer, prompt_2)
        predicted_answer = extract_answer(raw_output_2)

        # --- ÉVALUATION ET SAUVEGARDE ---
        metrics = calculate_metrics(row['gold_ids'], predicted_ids)

        results.append({
            "sample_id": row['id'],
            "question": row['question'],
            "gold_ids": row['gold_ids'],
            "predicted_ids": predicted_ids,
            "predicted_answer": predicted_answer,
            **metrics
        })

        print(f"--- SAMPLE {index+1} (ID: {row['id']}) ---")
        print(f"Question : {row['question']}")
        print(f"Réponse  : {predicted_answer}")
        print(f"Gold IDs : {row['gold_ids']}")
        print(f"Pred IDs : {predicted_ids}")
        print(f"Verdict  : {'✅ Exact' if metrics['accuracy'] else '❌ Différent'} (P: {metrics['precision']:.2f}, R: {metrics['recall']:.2f}, F1: {metrics['f1']:.2f})")
        print("-" * 50)

    # Agrégation des résultats finaux
    df_results = pd.DataFrame(results)
    avg_metrics = df_results[["precision", "recall", "f1", "accuracy"]].mean().to_dict()

    print("\n" + "="*20 + " RÉSULTATS APPROCHE 1 " + "="*20)
    for k, v in avg_metrics.items():
        print(f"Moyenne {k.capitalize()} : {v:.4f}")
    print("="*60 + "\n")

    if output_filename:
        df_results.to_csv(output_filename, index=False, encoding='utf-8')
        print(f" Résultats sauvegardés dans : {output_filename}")

    return df_results, avg_metrics

# Approche 2

In [ ]:
import pandas as pd

# On suppose que generate_llm_response, extract_answer, extract_ids,
# et calculate_metrics sont déjà chargées en mémoire.

def evaluate_approach_2(dataset, prompt_answer_after, prompt_chunks_after, model, tokenizer, sample_size=None, output_filename=None):
    """
    Évalue l'Approche 2 : LLM 1 répond -> LLM 2 audite la réponse pour trouver les IDs.
    """
    results = []
    df_sample = dataset.head(sample_size) if sample_size is not None else dataset

    for index, row in df_sample.iterrows():
        # --- PRÉPARATION DU CONTEXTE GLOBAL ---
        # Le contexte est généré une seule fois et utilisé pour les deux appels
        context_text = "\n".join([f"ID: {p['id']} | Paragraph: {p['text']}" for p in row['contexts']])

        # --- ÉTAPE 1 : Génération de la Réponse ---
        prompt_1 = prompt_answer_after.format(
            context_text=context_text,
            question=row['question']
        )

        raw_output_1 = generate_llm_response(model, tokenizer, prompt_1)
        predicted_answer = extract_answer(raw_output_1)

        # --- ÉTAPE 2 : Audit et Identification des Chunks ---
        # On injecte le même contexte ET la réponse prédite à l'étape 1
        prompt_2 = prompt_chunks_after.format(
            context_text=context_text,
            predicted_answer=predicted_answer
        )

        raw_output_2 = generate_llm_response(model, tokenizer, prompt_2)
        predicted_ids = extract_ids(raw_output_2)

        # --- ÉVALUATION ET SAUVEGARDE ---
        metrics = calculate_metrics(row['gold_ids'], predicted_ids)

        results.append({
            "sample_id": row['id'],
            "question": row['question'],
            "gold_ids": row['gold_ids'],
            "predicted_ids": predicted_ids,
            "predicted_answer": predicted_answer,
            **metrics
        })

        # Affichage visuel
        print(f"--- SAMPLE {index+1} (ID: {row['id']}) ---")
        print(f"Question : {row['question']}")
        print(f"Réponse  : {predicted_answer}")
        print(f"Gold IDs : {row['gold_ids']}")
        print(f"Pred IDs : {predicted_ids}")
        print(f"Verdict  : {'✅ Exact' if metrics['accuracy'] else '❌ Différent'} (P: {metrics['precision']:.2f}, R: {metrics['recall']:.2f}, F1: {metrics['f1']:.2f})")
        print("-" * 50)

    # Agrégation des résultats finaux
    df_results = pd.DataFrame(results)
    avg_metrics = df_results[["precision", "recall", "f1", "accuracy"]].mean().to_dict()

    print("\n" + "="*20 + " RÉSULTATS APPROCHE 2 " + "="*20)
    for k, v in avg_metrics.items():
        print(f"Moyenne {k.capitalize()} : {v:.4f}")
    print("="*60 + "\n")

    # Sauvegarde conditionnelle
    if output_filename:
        df_results.to_csv(output_filename, index=False, encoding='utf-8')
        print(f" Résultats sauvegardés dans : {output_filename}")

    return df_results, avg_metrics

In [ ]:
# Lancement de l'évaluation pour l'Approche 2
df_res_app2, metrics_app2 = evaluate_approach_2(
    dataset=ds_rag,
    prompt_answer_after=answer_chunks_after, # Ton 1er prompt (génère la réponse)
    prompt_chunks_after=chunks_after,        # Ton 2ème prompt (audite et cite)
    model=model,
    tokenizer=tokenizer,
    sample_size=20,                          # La taille de l'échantillon de test
    output_filename="resultats_approche2_llama3.csv"
)

# Optionnel : Afficher les premières lignes pour vérifier que tout est bien structuré
display(df_res_app2.head())

# Approche 5